In [5]:
import os
import pandas as pd

In [14]:
# Parse an autodam durability schedule (.sch) file.
# Each entry line is: *glob_pattern*  repeats  weight

from pathlib import Path
import re


def _find_repo_root() -> Path:
    start = Path.cwd().resolve()
    for d in [start, *start.parents]:
        if (d / "server" / "pyproject.toml").is_file():
            return d
    return start


def parse_autodam_schedule(path: Path) -> dict:
    """Parse an autodam schedule file into metadata and glob-pattern entries."""
    path = path.expanduser().resolve()
    if not path.exists():
        raise FileNotFoundError(f"File not found: {path}")

    schedule = {
        "file": str(path),
        "schedule_id": None,
        "multiplier": 1.0,
        "entries": [],
    }

    entry_re = re.compile(r"^\*([^*]+)\*\s+(\d+)\s+([\d.]+)\s*$")
    id_re = re.compile(r"^\*id\s+(.+)$", re.IGNORECASE)
    mult_re = re.compile(r"^\*multiplier\s+([\d.]+)\s*$", re.IGNORECASE)

    for raw_line in path.read_text(errors="replace").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#"):
            continue

        if m := id_re.match(line):
            schedule["schedule_id"] = m.group(1).strip()
            continue
        if m := mult_re.match(line):
            schedule["multiplier"] = float(m.group(1))
            continue
        if line.lower().startswith("*summary"):
            break
        if m := entry_re.match(line):
            schedule["entries"].append(
                {
                    "pattern": m.group(1),
                    "repeats": int(m.group(2)),
                    "weight": float(m.group(3)),
                }
            )

    return schedule


sch_path = _find_repo_root() / "data/raw/13999/v58_data_processing/gmw17287_95per_bt1cc_iver.sch"
schedule = parse_autodam_schedule(sch_path)
schedule_df = pd.DataFrame(schedule["entries"])

print(f"File: {schedule['file']}")
print(f"Schedule ID: {schedule['schedule_id']}")
print(f"Global multiplier: {schedule['multiplier']}")
print(f"Entries: {len(schedule_df)}")
schedule_df.head(5)

File: /data/home/tkodippili/Desktop/localTest_Analysis_DashboardV3/Dashboard/data/raw/13999/v58_data_processing/gmw17287_95per_bt1cc_iver.sch
Schedule ID: gmw17287 2WD SUV VRLDA Major Durability 150,000 miles for 95th percentile
Global multiplier: 1.0
Entries: 60


,pattern,repeats,weight
0,4e1,16,0.75
1,mf4e3_100,16,0.15
2,mf4e3_400,16,0.10
3,4f1,8,1.00
4,4x1,1200,0.75


In [22]:
def get_rsp_csv_file_info(folder_path):
    """
    List .csv and .rsp files in a folder and return a DataFrame with columns for file path and filename.

    Parameters
    ----------
    folder_path : str
        Path to the folder containing CSV/RSP files

    Returns
    -------
    pd.DataFrame
        DataFrame with columns:
        - col1: file path (relative or absolute)
        - col2: base file name
    """
    import glob

    data_files = []
    for ext in ("*.csv", "*.rsp"):
        data_files.extend(glob.glob(os.path.join(folder_path, ext)))
    data_files = sorted(set(data_files))

    file_info = []

    for data_file in data_files:
        file_info.append(
            {
                "file_path": data_file,
                "file_name": os.path.basename(data_file),
            }
        )

    result_df = pd.DataFrame(file_info)
    if result_df.empty:
        return result_df
    return result_df.reset_index(drop=True)

# run func
folder_path = "../data/raw/13999/v58_data_processing"
filename_df = get_rsp_csv_file_info(folder_path)
filename_df.head(5) 


,file_path,file_name
0,../data/raw/13999/v58_data_processing/mf4e1_bt...,mf4e1_bt1cc_coil_2m24_lt27550r22_5dec22_lca_lr...
1,../data/raw/13999/v58_data_processing/mf4e3_10...,mf4e3_100_bt1cc_coil_2m24_lt27550r22_5dec22_lc...
2,../data/raw/13999/v58_data_processing/mf4e3_40...,mf4e3_400_bt1cc_coil_2m24_lt27550r22_5dec22_lc...
3,../data/raw/13999/v58_data_processing/mf4f1_bt...,mf4f1_bt1cc_coil_2m24_lt27550r22_5dec22_lca_lr...
4,../data/raw/13999/v58_data_processing/mf4x1_bt...,mf4x1_bt1cc_coil_2m24_lt27550r22_5dec22_lca_lr...


In [33]:
import pandas as pd

dcy_df = pd.DataFrame(columns=["rsp_file_name", "rsp_event_name", "schedule_sequence", "repeats"])


In [ ]:
# Add file names from filename_df to dcy_df under the "rsp_file_name" column
dcy_df = pd.DataFrame({"rsp_file_name": filename_df["file_name"]})


def match_schedule_file_names(dcy_df, schedule_df):
    """Match each RSP filename to a schedule glob pattern and enrich with schedule metadata."""
    patterns = schedule_df["pattern"].tolist()

    def find_pattern(rsp_file_name):
        stem = os.path.splitext(rsp_file_name)[0]
        matches = [p for p in patterns if p in stem]
        return max(matches, key=len) if matches else None

    def event_name(rsp_file_name):
        stem = os.path.splitext(rsp_file_name)[0]
        return stem.split("_bt1cc")[0] # THIS IS HARDCODED!!
 
    result = dcy_df.copy()
    result["rsp_event_name"] = result["rsp_file_name"].map(event_name)
    result["matched_schedule_pattern"] = result["rsp_file_name"].map(find_pattern)

    schedule_lookup = schedule_df.reset_index().rename(columns={"index": "schedule_sequence"})
    schedule_lookup["schedule_sequence"] = schedule_lookup["schedule_sequence"] + 1

    result = result.merge(
        schedule_lookup[["pattern", "repeats", "weight", "schedule_sequence"]],
        left_on="matched_schedule_pattern",
        right_on="pattern",
        how="left",
    ).drop(columns=["pattern"])

    return result


dcy_df = match_schedule_file_names(dcy_df, schedule_df)
dcy_df


,rsp_file_name,rsp_event_name,matched_schedule_pattern,repeats,weight,schedule_sequence
0,mf4e1_bt1cc_coil_2m24_lt27550r22_5dec22_lca_lr...,mf4e1,4e1,16,0.75,1
1,mf4e3_100_bt1cc_coil_2m24_lt27550r22_5dec22_lc...,mf4e3_100,mf4e3_100,16,0.15,2
2,mf4e3_400_bt1cc_coil_2m24_lt27550r22_5dec22_lc...,mf4e3_400,mf4e3_400,16,0.10,3
3,mf4f1_bt1cc_coil_2m24_lt27550r22_5dec22_lca_lr...,mf4f1,4f1,8,1.00,4
4,mf4x1_bt1cc_coil_2m24_lt27550r22_5dec22_lca_lr...,mf4x1,4x1,1200,0.75,5
5,mf4x3_100_bt1cc_coil_2m24_lt27550r22_5dec22_lc...,mf4x3_100,mf4x3_100,1200,0.15,6
6,mf4x3_400_bt1cc_coil_2m24_lt27550r22_5dec22_lc...,mf4x3_400,mf4x3_400,1200,0.10,7
7,mfbm1_bt1cc_coil_2m24_lt27550r22_5dec22_lca_lr...,mfbm1,bm1,4500,0.75,8
8,mfbm3_100_bt1cc_coil_2m24_lt27550r22_5dec22_lc...,mfbm3_100,mfbm3_100,4500,0.15,9
9,mfbm3_400_bt1cc_coil_2m24_lt27550r22_5dec22_lc...,mfbm3_400,mfbm3_400,4500,0.10,10
